In [ ]:
!pip install ipwhois dnspython

In [ ]:
"""
Objetivo: extrair as_number, aut_name, org_id, org_name, country
a partir do NS-A (IP do nameserver) de um domínio.
"""

import os
import time
import socket
import dns.resolver
from ipwhois import IPWhois

# ------------------------------
# Configurações
# ------------------------------
DNS_TIMEOUT = 3.0         # segundos por consulta
DNS_LIFETIME = 5.0        # tempo total antes de desistir
MAX_RETRIES = 2           # tentativas por consulta
RETRY_SLEEP = 0.3         # intervalo entre tentativas

# ------------------------------
# Utilidades de DNS
# ------------------------------

def _dns_query(name: str, rtype: str):
    """Consulta DNS com timeout e retries, retornando uma lista (eventualmente vazia)."""
    resolver = dns.resolver.Resolver()
    resolver.timeout = DNS_TIMEOUT
    resolver.lifetime = DNS_LIFETIME

    last_exc = None
    for _ in range(MAX_RETRIES + 1):
        try:
            ans = resolver.resolve(name, rtype)
            return list(ans)
        except Exception as e:
            last_exc = e
            time.sleep(RETRY_SLEEP)
    # Falhou
    return []

def resolve_ns(domain: str):
    """Resolve registros NS do domínio (lista de hostnames, sem ponto final)."""
    out = []
    for rr in _dns_query(domain, 'NS'):
        try:
            host = str(rr.target).rstrip('.')
        except AttributeError:
            host = str(rr).rstrip('.')
        out.append(host)
    return out

def resolve_first_ip(hostname: str):
    """
    Tenta resolver A e AAAA do hostname.
    Retorna (ipv4, ipv6) strings ou None.
    Preferência para IPv4 quando ambas existem.
    """
    ipv4 = None
    ipv6 = None
    # A
    for rr in _dns_query(hostname, 'A'):
        try:
            ipv4 = rr.to_text()
            break
        except Exception:
            ipv4 = str(rr)
            break
    # AAAA
    for rr in _dns_query(hostname, 'AAAA'):
        try:
            ipv6 = rr.to_text()
            break
        except Exception:
            ipv6 = str(rr)
            break
    return ipv4, ipv6

# ------------------------------
# Parser RDAP
# ------------------------------

def extract_org_from_rdap(result: dict):
    """
    Extrai (org_id, org_name) do RDAP retornado por ipwhois.
    Tenta primeiro objetos com papel 'registrant' ou 'organization',
    depois outros papéis comuns (admin, tech, abuse) como fallback.
    """
    org_id = None
    org_name = None

    # ipwhois costuma trazer objetos em result["objects"] = { handle: {...} }
    objects = result.get("objects", {}) or {}

    # Preferências de papéis para tentar pegar a organização
    role_priority = [
        "registrant", "organization", "registrar",
        "administrative", "technical", "abuse"
    ]

    # 1) varre por prioridade de papéis
    for role in role_priority:
        for handle, obj in objects.items():
            roles = obj.get("roles", []) or []
            if role in roles:
                contact = obj.get("contact", {}) or {}
                name = contact.get("name")
                if name and not org_name:
                    org_name = str(name).replace('"', '').replace("'", "")
                if handle and not org_id:
                    org_id = str(handle).strip()
                if org_id and org_name:
                    return org_id, org_name

    # 2) fallback: pega qualquer objeto com 'contact.name' e 'handle'
    if not (org_id and org_name):
        for handle, obj in objects.items():
            contact = obj.get("contact", {}) or {}
            name = contact.get("name")
            if name and not org_name:
                org_name = str(name).replace('"', '').replace("'", "")
            if handle and not org_id:
                org_id = str(handle).strip()
            if org_id and org_name:
                break

    return org_id, org_name

def rdap_from_ip(ip: str):
    """
    Faz RDAP de um IP e retorna dicionário padronizado com:
    as_number, aut_name, org_id, org_name, country, registry (opcional).
    """
    try:
        ipwho = IPWhois(ip)
        # lookup_rdap é o mais completo; ipwhois faz cache por processo.
        res = ipwho.lookup_rdap(depth=1)
    except Exception:
        return {
            "as_number": None, "aut_name": None,
            "org_id": None, "org_name": None,
            "country": None, "registry": None
        }

    as_number  = res.get("asn")
    aut_name   = res.get("asn_description")
    registry   = res.get("asn_registry")
    country    = res.get("asn_country_code")

    # limpa aut_name (muitos vêm "DESC, algo" — pegue só a parte principal)
    if isinstance(aut_name, str):
        aut_name = aut_name.split(",")[0].strip()

    org_id, org_name = extract_org_from_rdap(res)

    return {
        "as_number": as_number,
        "aut_name": aut_name,
        "org_id": org_id,
        "org_name": org_name,
        "country": country,
        "registry": registry
    }

# ------------------------------
# Função principal por domínio
# ------------------------------

def get_ns_info(domain: str):
    """
    Para um domínio:
      - resolve NS,
      - resolve A/AAAA do NS,
      - consulta RDAP no IP (prefere IPv4),
      - retorna lista de dicts com campos solicitados.
    """
    domain = str(domain).strip()
    rows = []

    try:
        ns_hosts = resolve_ns(domain)
    except Exception as e:
        # Falha geral de NS: devolve uma linha com erro
        return [{
            "domain": domain, "NS": None, "NS-A": None, "NS-AAAA": None,
            "as_number": None, "aut_name": None, "org_id": None,
            "org_name": None, "country": None, "registry": None,
            "error": f"NS lookup failed: {e}"
        }]

    if not ns_hosts:
        return [{
            "domain": domain, "NS": None, "NS-A": None, "NS-AAAA": None,
            "as_number": None, "aut_name": None, "org_id": None,
            "org_name": None, "country": None, "registry": None,
            "error": "No NS records"
        }]

    for ns in ns_hosts:
        ipv4, ipv6 = resolve_first_ip(ns)

        # prioridade IPv4; se não houver, usa IPv6
        ip_for_rdap = ipv4 or ipv6

        rdap_info = {
            "as_number": None, "aut_name": None, "org_id": None,
            "org_name": None, "country": None, "registry": None
        }
        if ip_for_rdap:
            rdap_info = rdap_from_ip(ip_for_rdap)

        rows.append({
            "domain": domain,
            "NS": ns,
            "NS-A": ipv4,
            "NS-AAAA": ipv6,
            "as_number": rdap_info.get("as_number"),
            "aut_name": rdap_info.get("aut_name"),
            "org_id": rdap_info.get("org_id"),
            "org_name": rdap_info.get("org_name"),
            "country": rdap_info.get("country"),
            "registry": rdap_info.get("registry"),
        })

    return rows

# ------------------------------
# (Opcional) utilidade direta por IP
# ------------------------------

def get_info_from_ip(ip: str):
    """Para testes pontuais quando você já tem NS-A (IP) sem passar por DNS."""
    info = rdap_from_ip(ip)
    info.update({"NS-A": ip})
    return info

In [3]:
import os
import time
import math
import pandas as pd
from ipwhois import IPWhois

# =========================
# Configurações
# =========================
INPUT_CSV  = "./enriched_ns_batches_as_number_empty.csv"
OUT_CSV    = "./enriched_ns_batches22.csv"
ERR_CSV    = "./enriched_ns_errors.csv"
BATCH_SIZE = 50
SLEEP_BETWEEN_RDAP = 0.15  # para evitar rate limit dos RIRs
SLEEP_BETWEEN_BATCHES = 0.5

# Colunas finais exigidas (ordem exata)
FINAL_COLS = [
    "domain","NS","NS-A","company_name","revenue","country_name","sector","rank",
    "prefix","prefix_length","as_number","changed_x","aut_name","org_id","opaque_id",
    "source_x","changed_y","org_name","country","source_y"
]

# =========================
# Funções auxiliares
# =========================
def ensure_output_headers():
    """Cria os arquivos de saída com cabeçalhos se ainda não existem."""
    if not os.path.exists(OUT_CSV):
        pd.DataFrame(columns=FINAL_COLS).to_csv(OUT_CSV, index=False)
    if not os.path.exists(ERR_CSV):
        pd.DataFrame(columns=["domain","NS","NS-A","error"]).to_csv(ERR_CSV, index=False)

def rdap_from_ip(ip: str) -> dict:
    """
    Consulta RDAP para um IP (IPv4/IPv6) e retorna campos padronizados.
    Campos retornados:
      as_number, aut_name, org_id, org_name, country, registry
    """
    out = {
        "as_number": None,
        "aut_name": None,
        "org_id": None,
        "org_name": None,
        "country": None,
        "registry": None
    }
    if not ip or pd.isna(ip):
        return out

    try:
        ipw = IPWhois(str(ip).strip())
        res = ipw.lookup_rdap(depth=1)
    except Exception:
        return out

    # ASN
    out["as_number"] = res.get("asn")
    out["aut_name"]  = res.get("asn_description")
    out["registry"]  = res.get("asn_registry")
    out["country"]   = res.get("asn_country_code")

    # Limpeza do aut_name (pegar o primeiro trecho antes de vírgula, quando existir)
    if isinstance(out["aut_name"], str):
        out["aut_name"] = out["aut_name"].split(",")[0].strip()

    # Pegar org_id e org_name dos objects
    org_id, org_name = None, None
    objects = res.get("objects") or {}
    # tentar prioridade de roles
    role_priority = ["registrant", "organization", "registrar", "administrative", "technical", "abuse"]
    found = False
    for role in role_priority:
        if found:
            break
        for handle, obj in objects.items():
            roles = obj.get("roles") or []
            if role in roles:
                contact = obj.get("contact") or {}
                name = contact.get("name")
                if name and not org_name:
                    org_name = str(name).replace('"', '').replace("'", "")
                if handle and not org_id:
                    org_id = str(handle).strip()
                if org_id and org_name:
                    found = True
                    break

    # fallback: qualquer objeto com handle/name
    if not (org_id and org_name):
        for handle, obj in objects.items():
            contact = obj.get("contact") or {}
            name = contact.get("name")
            if name and not org_name:
                org_name = str(name).replace('"', '').replace("'", "")
            if handle and not org_id:
                org_id = str(handle).strip()
            if org_id and org_name:
                break

    out["org_id"] = org_id
    out["org_name"] = org_name
    return out

def empty_if_missing(row_dict: dict, cols: list) -> dict:
    """Garante que todas as colunas finais existem; faltantes recebem ''. Mantém ordem."""
    out = {}
    for c in cols:
        out[c] = row_dict.get(c, "")
        if pd.isna(out[c]):
            out[c] = ""
    return out

# =========================
# Execução
# =========================
def main():
    ensure_output_headers()

    # Ler input (forçar texto p/ não perder zeros ou números grandes)
    df = pd.read_csv(
        INPUT_CSV,
        dtype=str,
        keep_default_na=False  # strings vazias ao invés de NaN (mais previsível ao escrever CSV final)
    )

    # Normalizar colunas esperadas pela especificação de entrada
    expected_in = ["domain","NS","NS-A","company_name","revenue","country_name","sector","rank","prefix_norm","prefix","prefix_length","as_number"]
    for col in expected_in:
        if col not in df.columns:
            df[col] = ""

    total = len(df)
    num_batches = math.ceil(total / BATCH_SIZE)
    print(f"Total linhas: {total} | Batches: {num_batches} (tam={BATCH_SIZE})")

    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = min((i + 1) * BATCH_SIZE, total)
        chunk = df.iloc[start:end].copy()

        out_rows = []
        err_rows = []

        print(f"[Batch {i+1}/{num_batches}] Processando linhas {start}..{end-1}")

        for idx, row in chunk.iterrows():
            domain = row.get("domain", "")
            ns     = row.get("NS", "")
            ns_a   = row.get("NS-A", "")

            try:
                rdap = rdap_from_ip(ns_a)

                # Monta a linha final conforme ordem e nomes especificados
                final_row = {
                    "domain": domain,
                    "NS": ns,
                    "NS-A": ns_a,
                    "company_name": row.get("company_name", ""),
                    "revenue": row.get("revenue", ""),
                    "country_name": row.get("country_name", ""),
                    "sector": row.get("sector", ""),
                    "rank": row.get("rank", ""),
                    "prefix": row.get("prefix", ""),
                    "prefix_length": row.get("prefix_length", ""),
                    "as_number": rdap.get("as_number", ""),
                    "changed_x": "",            # não fornecido -> vazio
                    "aut_name": rdap.get("aut_name", ""),
                    "org_id": rdap.get("org_id", ""),
                    "opaque_id": "",            # não fornecido -> vazio
                    "source_x": "",             # não fornecido -> vazio
                    "changed_y": "",            # não fornecido -> vazio
                    "org_name": rdap.get("org_name", ""),
                    "country": rdap.get("country", ""),
                    "source_y": ""              # não fornecido -> vazio
                }

                out_rows.append(empty_if_missing(final_row, FINAL_COLS))

            except Exception as e:
                # Registra erro
                err_rows.append({
                    "domain": domain,
                    "NS": ns,
                    "NS-A": ns_a,
                    "error": str(e)
                })

            # pequena pausa para evitar throttling
            time.sleep(SLEEP_BETWEEN_RDAP)

        # Salva parciais (append)
        if out_rows:
            pd.DataFrame(out_rows, columns=FINAL_COLS).to_csv(OUT_CSV, mode="a", index=False, header=False)
        if err_rows:
            pd.DataFrame(err_rows, columns=["domain","NS","NS-A","error"]).to_csv(ERR_CSV, mode="a", index=False, header=False)

        print(f"[Batch {i+1}] salvos: {len(out_rows)} ok | {len(err_rows)} erros")
        time.sleep(SLEEP_BETWEEN_BATCHES)

    print("Concluído.")

if __name__ == "__main__":
    main()

Total linhas: 127 | Batches: 3 (tam=50)
[Batch 1/3] Processando linhas 0..49
[Batch 1] salvos: 50 ok | 0 erros
[Batch 2/3] Processando linhas 50..99
[Batch 2] salvos: 50 ok | 0 erros
[Batch 3/3] Processando linhas 100..126
[Batch 3] salvos: 27 ok | 0 erros
Concluído.


In [2]:
import pandas as pd

#filtrar linhas de enriched_ns_batches.csv onde as_number está vazio
df_enriched_ns = pd.read_csv('./enriched_ns_batches22.csv')
df_enriched_ns_filtered = df_enriched_ns[df_enriched_ns['as_number'].isnull()]

df_enriched_ns_filtered.to_csv('./enriched_ns_batches_as_number_empty.csv', index=False)

#excluir de enriched_ns_batches.csv as linhas que estão em enriched_ns_batches_as_number_empty.csv
df_enriched_ns = df_enriched_ns[~df_enriched_ns['domain'].isin(df_enriched_ns_filtered['domain'])]
df_enriched_ns.to_csv('./enriched_ns_batches_done2.csv', index=False)
